# ReBeL HUNL training on Colab H100

Select **Runtime > Change runtime type > H100 GPU**. This notebook runs a smoke test first, then exposes production commands. The implementation is an open approximation of ReBeL, not Meta's private poker checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/henkeskevin/rebel.git /content/rebel
%cd /content/rebel
!pip install -q -r requirements-nlhe.txt

In [ ]:
import os, torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'Enable an H100 GPU runtime first'
ROOT = '/content/drive/MyDrive/rebel_hunl'
os.makedirs(ROOT, exist_ok=True)

## Smoke test

This validates generation, BF16-compatible model execution, checkpointing, evaluation, and inference in a few minutes.

In [ ]:
!python scripts/generate_nlhe_data.py --output /content/nlhe_smoke/data/river --street river --examples 16 --shard-size 8 --cfr-iterations 4 --search-depth 2 --workers 2
!python scripts/train_nlhe.py --data /content/nlhe_smoke/data --output /content/nlhe_smoke/run --profile smoke --device cuda --epochs 1 --batch-size 8 --no-compile
!python scripts/eval_nlhe.py --checkpoint /content/nlhe_smoke/run/latest.pt --data /content/nlhe_smoke/data --device cuda

## Production bootstrap

River labels use exact showdown values. Earlier streets use Monte Carlo check-down values for the first bootstrap. Data generation is CPU-heavy, so use all available Colab CPU workers.

In [ ]:
!python scripts/generate_nlhe_data.py --output "$ROOT/data/bootstrap/river" --street river --examples 20000 --shard-size 64 --cfr-iterations 128 --search-depth 6 --workers 8
!python scripts/generate_nlhe_data.py --output "$ROOT/data_validation/river" --street river --examples 1000 --shard-size 64 --cfr-iterations 128 --search-depth 6 --workers 8 --seed 9001
!python scripts/generate_nlhe_data.py --output "$ROOT/data/bootstrap/turn" --street turn --examples 10000 --shard-size 64 --cfr-iterations 64 --search-depth 4 --rollout-boards 4 --workers 8
!python scripts/generate_nlhe_data.py --output "$ROOT/data/bootstrap/flop" --street flop --examples 5000 --shard-size 64 --cfr-iterations 64 --search-depth 4 --rollout-boards 4 --workers 8
!python scripts/generate_nlhe_data.py --output "$ROOT/data/bootstrap/preflop" --street preflop --examples 2500 --shard-size 64 --cfr-iterations 32 --search-depth 3 --rollout-boards 2 --workers 8

In [ ]:
!python scripts/train_nlhe.py --data "$ROOT/data/bootstrap" --output "$ROOT/runs/bootstrap_h100" --profile h100 --device cuda --epochs 20 --batch-size 384

## Improvement iteration

Regenerate targets with the trained value network at depth-limited leaves, then resume training. Start with river and add the other streets after checking held-out metrics.

In [ ]:
!python scripts/generate_nlhe_data.py --output "$ROOT/data/iteration_01/river" --street river --examples 10000 --shard-size 64 --cfr-iterations 256 --search-depth 6 --checkpoint "$ROOT/runs/bootstrap_h100/latest.pt" --device cuda --workers 1
!python scripts/train_nlhe.py --data "$ROOT/data" --output "$ROOT/runs/iteration_01" --profile h100 --device cuda --epochs 30 --batch-size 384 --resume "$ROOT/runs/bootstrap_h100/latest.pt"

In [ ]:
!python scripts/eval_nlhe.py --checkpoint "$ROOT/runs/iteration_01/latest.pt" --data "$ROOT/data_validation/river" --device cuda
!python scripts/solve_nlhe.py --checkpoint "$ROOT/runs/iteration_01/latest.pt" --device cuda --iterations 256 --depth 4 --hand AsKd